In [1]:
# ==========================================================
# CELL 1 — IMPORTS AND PATHS
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Project directories
BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", BASE_DIR)
print("Data directory:", DATA_DIR)

Project directory: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics
Data directory: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\data


In [2]:
# ==========================================================
# CELL 2 — LOAD CLEANED DATA
# ==========================================================

players = pd.read_csv(
    INTERIM_DIR / "players_clean.csv"
)

teams = pd.read_csv(
    INTERIM_DIR / "teams_clean.csv"
)

matches = pd.read_csv(
    INTERIM_DIR / "matches_clean.csv"
)

print("Players:", players.shape)
print("Teams:", teams.shape)
print("Matches:", matches.shape)

Players: (1248, 70)
Teams: (48, 132)
Matches: (104, 42)


In [3]:
# ==========================================================
# CELL 3 — BASIC VALIDATION
# ==========================================================

print("=" * 70)
print("PLAYERS")
print("=" * 70)

print(players.shape)
print(players.columns.tolist())

print("\n")

print("=" * 70)
print("TEAMS")
print("=" * 70)

print(teams.shape)
print(teams.columns.tolist())

print("\n")

print("=" * 70)
print("MATCHES")
print("=" * 70)

print(matches.shape)
print(matches.columns.tolist())

PLAYERS
(1248, 70)
['player', 'team', 'team_country', 'position', 'age', 'birth_year', 'club', 'games', 'games_starts', 'minutes', 'minutes_90s', 'goals', 'assists', 'goals_assists', 'goals_pens', 'pens_made', 'pens_att', 'cards_yellow', 'cards_red', 'goals_per90', 'assists_per90', 'goals_assists_per90', 'goals_pens_per90', 'goals_assists_pens_per90', 'shots', 'shots_on_target', 'shots_on_target_pct', 'shots_per90', 'shots_on_target_per90', 'goals_per_shot', 'goals_per_shot_on_target', 'minutes_per_game', 'minutes_pct', 'minutes_per_start', 'games_complete', 'games_subs', 'minutes_per_sub', 'unused_subs', 'points_per_game', 'on_goals_for', 'on_goals_against', 'plus_minus', 'plus_minus_per90', 'plus_minus_wowy', 'cards_yellow_red', 'fouls', 'fouled', 'offsides', 'crosses', 'interceptions', 'tackles_won', 'own_goals', 'gk_games', 'gk_games_starts', 'gk_minutes', 'gk_goals_against', 'gk_goals_against_per90', 'gk_shots_on_target_against', 'gk_saves', 'gk_save_pct', 'gk_wins', 'gk_ties', 'g

In [4]:
# ==========================================================
# CELL 4 — PLAYER GOAL CONTRIBUTION
# ==========================================================

players_fe = players.copy()

players_fe["goals"] = players_fe["goals"].fillna(0)
players_fe["assists"] = players_fe["assists"].fillna(0)

players_fe["goal_contributions"] = (
    players_fe["goals"]
    +
    players_fe["assists"]
)

print(
    players_fe[
        [
            "player",
            "team",
            "goals",
            "assists",
            "goal_contributions"
        ]
    ].head(10)
)

             player     team  goals  assists  goal_contributions
0      Achref Abada  Algeria    0.0      0.0                 0.0
1     Adil Boulbina  Algeria    0.0      0.0                 0.0
2      Amine Gouiri  Algeria    1.0      0.0                 1.0
3  Anis Hadj Moussa  Algeria    0.0      0.0                 0.0
4       Aïssa Mandi  Algeria    0.0      0.0                 0.0
5      Fares Chaïbi  Algeria    0.0      0.0                 0.0
6   Farés Ghedjemis  Algeria    0.0      0.0                 0.0
7   Hicham Boudaoui  Algeria    0.0      0.0                 0.0
8     Houssem Aouar  Algeria    0.0      2.0                 2.0
9      Ibrahim Maza  Algeria    0.0      0.0                 0.0


In [5]:
# ==========================================================
# CELL 5 — GOAL CONTRIBUTIONS PER 90
# ==========================================================

players_fe["goal_contributions_per90"] = (
    players_fe["goals_per90"].fillna(0)
    +
    players_fe["assists_per90"].fillna(0)
)

display(
    players_fe[
        [
            "player",
            "team",
            "goals_per90",
            "assists_per90",
            "goal_contributions_per90"
        ]
    ]
    .sort_values(
        "goal_contributions_per90",
        ascending=False
    )
    .head(20)
)

,player,team,goals_per90,assists_per90,goal_contributions_per90
1153,Kaan Ayhan,Türkiye,30.00,0.00,30.00
1086,Mattias Svanberg,Sweden,5.63,0.00,5.63
32,Flaco López,Argentina,0.00,4.50,4.50
136,Dennis Hadžikadunić,Bosnia–Herz,0.00,3.33,3.33
500,Deniz Undav,Germany,1.79,1.19,2.98
207,Yannick,Cabo Verde,0.00,2.81,2.81
140,Ermin Mahmić,Bosnia–Herz,2.65,0.00,2.65
1039,Tshepang Moremi,South Africa,0.00,2.65,2.65
893,Gonçalo Ramos,Portugal,2.50,0.00,2.50
1148,Can Uzun,Türkiye,0.00,2.37,2.37


In [6]:
# ==========================================================
# CELL 6 — SHOOTING EFFICIENCY
# ==========================================================

players_fe["shooting_efficiency"] = np.where(
    players_fe["shots"].fillna(0) > 0,
    players_fe["goals"].fillna(0)
    /
    players_fe["shots"],
    0
)

players_fe["shooting_efficiency"] = (
    players_fe["shooting_efficiency"]
    .round(4)
)

display(
    players_fe[
        [
            "player",
            "team",
            "shots",
            "goals",
            "shooting_efficiency"
        ]
    ]
    .sort_values(
        "shooting_efficiency",
        ascending=False
    )
    .head(20)
)

,player,team,shots,goals,shooting_efficiency
1153,Kaan Ayhan,Türkiye,1.0,1.0,1.0000
34,Giovani Lo Celso,Argentina,1.0,1.0,1.0000
1137,Omar Rekik,Tunisia,1.0,1.0,1.0000
821,Marcus Pedersen,Norway,1.0,1.0,1.0000
921,Hassan Al-Haydos,Qatar,1.0,1.0,1.0000
757,Crysencio Summerville,Netherlands,2.0,2.0,1.0000
742,Issa Diop,Morocco,1.0,1.0,1.0000
632,Kaishū Sano,Japan,1.0,1.0,1.0000
309,Petar Musa,Croatia,1.0,1.0,1.0000
440,Yasser Ibrahim,Egypt,1.0,1.0,1.0000


In [7]:
# ==========================================================
# CELL 7 — SHOT ACCURACY
# ==========================================================

players_fe["shot_accuracy"] = np.where(
    players_fe["shots"].fillna(0) > 0,
    players_fe["shots_on_target"].fillna(0)
    /
    players_fe["shots"],
    0
)

players_fe["shot_accuracy"] = (
    players_fe["shot_accuracy"]
    .round(4)
)

display(
    players_fe[
        [
            "player",
            "team",
            "shots",
            "shots_on_target",
            "shot_accuracy"
        ]
    ]
    .sort_values(
        "shot_accuracy",
        ascending=False
    )
    .head(20)
)

,player,team,shots,shots_on_target,shot_accuracy
1120,Anis Ben Slimane,Tunisia,1.0,1.0,1.0
16,Nadhir Benbouali,Algeria,2.0,2.0,1.0
1068,Anthony Elanga,Sweden,3.0,3.0,1.0
1070,Besfort Zeneli,Sweden,1.0,1.0,1.0
1153,Kaan Ayhan,Türkiye,1.0,1.0,1.0
1166,Zeki Çelik,Türkiye,1.0,1.0,1.0
1137,Omar Rekik,Tunisia,1.0,1.0,1.0
34,Giovani Lo Celso,Argentina,1.0,1.0,1.0
1106,Manuel Akanji,Switzerland,1.0,1.0,1.0
921,Hassan Al-Haydos,Qatar,1.0,1.0,1.0


In [8]:
# ==========================================================
# CELL 8 — DEFENSIVE ACTIONS
# ==========================================================

players_fe["tackles_won"] = (
    players_fe["tackles_won"].fillna(0)
)

players_fe["interceptions"] = (
    players_fe["interceptions"].fillna(0)
)

players_fe["defensive_actions"] = (
    players_fe["tackles_won"]
    +
    players_fe["interceptions"]
)

display(
    players_fe[
        [
            "player",
            "team",
            "position",
            "tackles_won",
            "interceptions",
            "defensive_actions"
        ]
    ]
    .sort_values(
        "defensive_actions",
        ascending=False
    )
    .head(20)
)

,player,team,position,tackles_won,interceptions,defensive_actions
472,Dayot Upamecano,France,DF,12.0,15.0,27.0
449,Elliot Anderson,England,MF,18.0,8.0,26.0
859,Andrés Cubas,Paraguay,MF,13.0,11.0,24.0
1060,Rodri,Spain,MF,20.0,3.0,23.0
26,Alexis Mac Allister,Argentina,MF,12.0,9.0,21.0
27,Cristian Romero,Argentina,DF,10.0,11.0,21.0
469,Aurélien Tchouaméni,France,MF,14.0,6.0,20.0
432,Mohanad Lasheen,Egypt,MF,11.0,8.0,19.0
1021,Khuliso Mudau,South Africa,DF,8.0,11.0,19.0
1059,Pedro Porro,Spain,DF,11.0,8.0,19.0


In [9]:
# ==========================================================
# CELL 9 — DEFENSIVE ACTIONS PER 90
# ==========================================================

players_fe["defensive_actions_per90"] = np.where(
    players_fe["minutes_90s"].fillna(0) > 0,
    players_fe["defensive_actions"]
    /
    players_fe["minutes_90s"],
    0
)

players_fe["defensive_actions_per90"] = (
    players_fe["defensive_actions_per90"]
    .round(3)
)

display(
    players_fe[
        [
            "player",
            "team",
            "position",
            "defensive_actions",
            "defensive_actions_per90"
        ]
    ]
    .sort_values(
        "defensive_actions_per90",
        ascending=False
    )
    .head(20)
)

,player,team,position,defensive_actions,defensive_actions_per90
661,Mohammad Abu Hasheesh,Jordan,DF,2.0,20.000
273,Joris Kayembe,Congo DR,"DF,FW",8.0,16.000
646,Yuito Suzuki,Japan,"FW,MF",1.0,10.000
672,Salim Obaid,Jordan,DF,2.0,10.000
576,Amirhossein Hosseinzadeh,IR Iran,FW,2.0,10.000
96,Patrick Wimmer,Austria,"FW,MF",2.0,10.000
1086,Mattias Svanberg,Sweden,MF,2.0,10.000
990,Assane Diao,Senegal,MF,1.0,10.000
518,Pascal Groß,Germany,"DF,MF",2.0,10.000
784,Callan Elliot,New Zealand,DF,1.0,10.000


In [10]:
# ==========================================================
# CELL 10 — PLAYER USAGE
# ==========================================================

players_fe["starts_ratio"] = np.where(
    players_fe["games"].fillna(0) > 0,
    players_fe["games_starts"].fillna(0)
    /
    players_fe["games"],
    0
)

players_fe["starts_ratio"] = (
    players_fe["starts_ratio"]
    .round(3)
)

display(
    players_fe[
        [
            "player",
            "team",
            "games",
            "games_starts",
            "starts_ratio",
            "minutes"
        ]
    ]
    .sort_values(
        "minutes",
        ascending=False
    )
    .head(20)
)

,player,team,games,games_starts,starts_ratio,minutes
28,Emiliano Martínez,Argentina,8,8,1.000,810.0
1057,Pau Cubarsí,Spain,8,8,1.000,750.0
1061,Unai Simón,Spain,8,8,1.000,750.0
1050,Marc Cucurella,Spain,8,8,1.000,750.0
26,Alexis Mac Allister,Argentina,8,7,0.875,750.0
41,Lionel Messi,Argentina,8,7,0.875,740.0
1040,Aymeric Laporte,Spain,8,8,1.000,727.0
1060,Rodri,Spain,8,8,1.000,724.0
486,Mike Maignan,France,8,8,1.000,720.0
477,Kylian Mbappé,France,8,8,1.000,695.0


In [11]:
# ==========================================================
# CELL 11 — PLAYER PERFORMANCE INDEX
# ==========================================================

players_fe["attacking_output"] = (
    players_fe["goals_per90"].fillna(0)
    +
    players_fe["assists_per90"].fillna(0)
)

players_fe["defensive_output"] = (
    players_fe["defensive_actions_per90"].fillna(0)
)

players_fe["performance_index"] = (
    players_fe["attacking_output"] * 0.6
    +
    players_fe["defensive_output"] * 0.4
)

players_fe["performance_index"] = (
    players_fe["performance_index"]
    .round(3)
)

display(
    players_fe[
        [
            "player",
            "team",
            "position",
            "attacking_output",
            "defensive_output",
            "performance_index"
        ]
    ]
    .sort_values(
        "performance_index",
        ascending=False
    )
    .head(20)
)

,player,team,position,attacking_output,defensive_output,performance_index
1153,Kaan Ayhan,Türkiye,"DF,MF",30.00,0.000,18.000
661,Mohammad Abu Hasheesh,Jordan,DF,0.00,20.000,8.000
1086,Mattias Svanberg,Sweden,MF,5.63,10.000,7.378
273,Joris Kayembe,Congo DR,"DF,FW",0.00,16.000,6.400
32,Flaco López,Argentina,FW,4.50,5.000,4.700
207,Yannick,Cabo Verde,MF,2.81,7.500,4.686
136,Dennis Hadžikadunić,Bosnia–Herz,DF,3.33,6.667,4.665
633,Keisuke Gotō,Japan,FW,0.00,10.000,4.000
96,Patrick Wimmer,Austria,"FW,MF",0.00,10.000,4.000
518,Pascal Groß,Germany,"DF,MF",0.00,10.000,4.000


In [12]:
# ==========================================================
# CELL 12 — TEAM GOAL EFFICIENCY
# ==========================================================

teams_fe = teams.copy()

teams_fe["goals"] = teams_fe["goals"].fillna(0)
teams_fe["shots"] = teams_fe["shots"].fillna(0)

teams_fe["goals_per_shot"] = np.where(
    teams_fe["shots"] > 0,
    teams_fe["goals"]
    /
    teams_fe["shots"],
    0
)

teams_fe["goals_per_shot"] = (
    teams_fe["goals_per_shot"]
    .round(4)
)

display(
    teams_fe[
        [
            "team",
            "goals",
            "shots",
            "goals_per_shot"
        ]
    ]
    .sort_values(
        "goals_per_shot",
        ascending=False
    )
)

,team,goals,shots,goals_per_shot
24,Japan,8,34,0.2353
29,Netherlands,10,46,0.2174
31,Norway,12,66,0.1818
17,England,20,118,0.1695
11,Croatia,6,37,0.1622
1,Argentina,18,114,0.1579
3,Austria,5,32,0.1562
45,United States,9,59,0.1525
19,Germany,11,74,0.1486
41,Sweden,7,48,0.1458


In [13]:
# ==========================================================
# CELL 13 — TEAM SHOT ACCURACY
# ==========================================================

teams_fe["shot_accuracy"] = np.where(
    teams_fe["shots"] > 0,
    teams_fe["shots_on_target"].fillna(0)
    /
    teams_fe["shots"],
    0
)

teams_fe["shot_accuracy"] = (
    teams_fe["shot_accuracy"]
    .round(4)
)

display(
    teams_fe[
        [
            "team",
            "shots",
            "shots_on_target",
            "shot_accuracy"
        ]
    ]
    .sort_values(
        "shot_accuracy",
        ascending=False
    )
)

,team,shots,shots_on_target,shot_accuracy
30,New Zealand,31,15,0.4839
41,Sweden,48,23,0.4792
29,Netherlands,46,22,0.4783
11,Croatia,37,17,0.4595
17,England,118,53,0.4492
31,Norway,66,29,0.4394
18,France,139,59,0.4245
36,Saudi Arabia,17,7,0.4118
42,Switzerland,74,30,0.4054
6,Brazil,74,30,0.4054


In [14]:
# ==========================================================
# CELL 14 — TEAM DEFENSIVE ACTIVITY
# ==========================================================

teams_fe["tackles_won"] = (
    teams_fe["tackles_won"].fillna(0)
)

teams_fe["interceptions"] = (
    teams_fe["interceptions"].fillna(0)
)

teams_fe["defensive_actions"] = (
    teams_fe["tackles_won"]
    +
    teams_fe["interceptions"]
)

display(
    teams_fe[
        [
            "team",
            "tackles_won",
            "interceptions",
            "defensive_actions"
        ]
    ]
    .sort_values(
        "defensive_actions",
        ascending=False
    )
)

,team,tackles_won,interceptions,defensive_actions
1,Argentina,93,82,175
40,Spain,88,64,152
18,France,77,65,142
33,Paraguay,88,53,141
17,England,77,51,128
31,Norway,67,44,111
42,Switzerland,52,52,104
45,United States,42,60,102
28,Morocco,61,40,101
16,Egypt,54,44,98


In [15]:
# ==========================================================
# CELL 15 — TEAM PERFORMANCE INDEX
# ==========================================================

teams_fe["attacking_efficiency"] = (
    teams_fe["goals_per90"].fillna(0)
)

teams_fe["defensive_efficiency"] = (
    teams_fe["plus_minus_per90"].fillna(0)
)

teams_fe["team_performance_index"] = (
    teams_fe["attacking_efficiency"] * 0.5
    +
    teams_fe["defensive_efficiency"] * 0.5
)

teams_fe["team_performance_index"] = (
    teams_fe["team_performance_index"]
    .round(3)
)

display(
    teams_fe[
        [
            "team",
            "attacking_efficiency",
            "defensive_efficiency",
            "team_performance_index"
        ]
    ]
    .sort_values(
        "team_performance_index",
        ascending=False
    )
)

,team,attacking_efficiency,defensive_efficiency,team_performance_index
19,Germany,2.54,1.38,1.960
18,France,2.50,1.25,1.875
29,Netherlands,2.31,1.38,1.845
27,Mexico,2.00,1.40,1.700
17,England,2.40,0.96,1.680
1,Argentina,2.00,1.22,1.610
6,Brazil,2.00,1.20,1.600
4,Belgium,2.05,1.11,1.580
40,Spain,1.56,1.56,1.560
24,Japan,2.00,0.75,1.375


In [16]:
# ==========================================================
# CELL 16 — MATCH FEATURES
# ==========================================================

matches_fe = matches.copy()

matches_fe["home_score"] = (
    matches_fe["home_score"].fillna(0)
)

matches_fe["away_score"] = (
    matches_fe["away_score"].fillna(0)
)

matches_fe["total_goals"] = (
    matches_fe["home_score"]
    +
    matches_fe["away_score"]
)

matches_fe["goal_difference"] = (
    matches_fe["home_score"]
    -
    matches_fe["away_score"]
)

matches_fe["result"] = np.select(
    [
        matches_fe["home_score"]
        >
        matches_fe["away_score"],

        matches_fe["home_score"]
        ==
        matches_fe["away_score"],

        matches_fe["home_score"]
        <
        matches_fe["away_score"]
    ],
    [
        "Home Win",
        "Draw",
        "Away Win"
    ],
    default="Unknown"
)

display(
    matches_fe[
        [
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "total_goals",
            "goal_difference",
            "result"
        ]
    ].head(20)
)

,home_team,away_team,home_score,away_score,total_goals,goal_difference,result
0,Mexico,South Africa,2.0,0.0,2.0,2.0,Home Win
1,Korea Republic,Czechia,2.0,1.0,3.0,1.0,Home Win
2,Canada,Bosnia–Herz,1.0,1.0,2.0,0.0,Draw
3,United States,Paraguay,4.0,1.0,5.0,3.0,Home Win
4,Qatar,Switzerland,1.0,1.0,2.0,0.0,Draw
5,Brazil,Morocco,1.0,1.0,2.0,0.0,Draw
6,Australia,Türkiye,2.0,0.0,2.0,2.0,Home Win
7,Haiti,Scotland,0.0,1.0,1.0,-1.0,Away Win
8,Germany,Curaçao,7.0,1.0,8.0,6.0,Home Win
9,Netherlands,Japan,2.0,2.0,4.0,0.0,Draw


In [17]:
# ==========================================================
# CELL 17 — POSSESSION DIFFERENCE
# ==========================================================

matches_fe["possession_difference"] = (
    matches_fe["home_possession"]
    -
    matches_fe["away_possession"]
)

display(
    matches_fe[
        [
            "home_team",
            "away_team",
            "home_possession",
            "away_possession",
            "possession_difference",
            "result"
        ]
    ].head(20)
)

,home_team,away_team,home_possession,away_possession,possession_difference,result
0,Mexico,South Africa,61,40,21,Home Win
1,Korea Republic,Czechia,62,38,24,Home Win
2,Canada,Bosnia–Herz,61,39,22,Draw
3,United States,Paraguay,65,35,30,Home Win
4,Qatar,Switzerland,32,68,-36,Draw
5,Brazil,Morocco,51,49,2,Draw
6,Australia,Türkiye,28,72,-44,Home Win
7,Haiti,Scotland,54,46,8,Away Win
8,Germany,Curaçao,65,35,30,Home Win
9,Netherlands,Japan,60,40,20,Draw


In [18]:
# ==========================================================
# CELL 18 — SHOT DIFFERENCE
# ==========================================================

matches_fe["shot_difference"] = (
    matches_fe["home_total_shots"]
    -
    matches_fe["away_total_shots"]
)

matches_fe["shots_on_target_difference"] = (
    matches_fe["home_sot"]
    -
    matches_fe["away_sot"]
)

display(
    matches_fe[
        [
            "home_team",
            "away_team",
            "home_total_shots",
            "away_total_shots",
            "shot_difference",
            "shots_on_target_difference"
        ]
    ].head(20)
)

,home_team,away_team,home_total_shots,away_total_shots,shot_difference,shots_on_target_difference
0,Mexico,South Africa,16,3,13,2
1,Korea Republic,Czechia,15,7,8,2
2,Canada,Bosnia–Herz,13,8,5,1
3,United States,Paraguay,16,9,7,5
4,Qatar,Switzerland,7,26,-19,-3
5,Brazil,Morocco,12,14,-2,2
6,Australia,Türkiye,9,30,-21,-4
7,Haiti,Scotland,15,9,6,0
8,Germany,Curaçao,26,8,18,10
9,Netherlands,Japan,10,10,0,3


In [19]:
# ==========================================================
# CELL 19 — MATCH INTENSITY
# ==========================================================

matches_fe["total_shots"] = (
    matches_fe["home_total_shots"]
    +
    matches_fe["away_total_shots"]
)

matches_fe["total_sot"] = (
    matches_fe["home_sot"]
    +
    matches_fe["away_sot"]
)

matches_fe["total_fouls"] = (
    matches_fe["home_fouls"]
    +
    matches_fe["away_fouls"]
)

matches_fe["total_yellow_cards"] = (
    matches_fe["home_cards_yellow"]
    +
    matches_fe["away_cards_yellow"]
)

matches_fe["match_intensity"] = (
    matches_fe["total_shots"]
    +
    matches_fe["total_fouls"]
)

display(
    matches_fe[
        [
            "home_team",
            "away_team",
            "total_goals",
            "total_shots",
            "total_sot",
            "total_fouls",
            "total_yellow_cards",
            "match_intensity"
        ]
    ].head(20)
)

,home_team,away_team,total_goals,total_shots,total_sot,total_fouls,total_yellow_cards,match_intensity
0,Mexico,South Africa,2.0,19,6,23,3,42
1,Korea Republic,Czechia,3.0,22,10,25,1,47
2,Canada,Bosnia–Herz,2.0,21,7,30,5,51
3,United States,Paraguay,5.0,25,7,30,6,55
4,Qatar,Switzerland,2.0,33,11,23,3,56
5,Brazil,Morocco,2.0,26,8,30,2,56
6,Australia,Türkiye,2.0,39,12,16,1,55
7,Haiti,Scotland,1.0,24,4,44,4,68
8,Germany,Curaçao,8.0,34,14,29,0,63
9,Netherlands,Japan,4.0,20,9,14,3,34


In [20]:
# ==========================================================
# CELL 20 — SAVE PLAYER FEATURES
# ==========================================================

players_fe.to_csv(
    PROCESSED_DIR / "players_features.csv",
    index=False
)

print(
    "Saved:",
    PROCESSED_DIR / "players_features.csv"
)

print(
    "Shape:",
    players_fe.shape
)

Saved: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\data\processed\players_features.csv
Shape: (1248, 80)


In [21]:
# ==========================================================
# CELL 21 — SAVE TEAM FEATURES
# ==========================================================

teams_fe.to_csv(
    PROCESSED_DIR / "teams_features.csv",
    index=False
)

print(
    "Saved:",
    PROCESSED_DIR / "teams_features.csv"
)

print(
    "Shape:",
    teams_fe.shape
)

Saved: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\data\processed\teams_features.csv
Shape: (48, 137)


In [22]:
# ==========================================================
# CELL 22 — SAVE MATCH FEATURES
# ==========================================================

matches_fe.to_csv(
    PROCESSED_DIR / "matches_features.csv",
    index=False
)

print(
    "Saved:",
    PROCESSED_DIR / "matches_features.csv"
)

print(
    "Shape:",
    matches_fe.shape
)

Saved: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\data\processed\matches_features.csv
Shape: (104, 53)


In [23]:
# ==========================================================
# CELL 23 — FEATURE ENGINEERING VALIDATION
# ==========================================================

print("=" * 70)
print("FEATURE ENGINEERING VALIDATION")
print("=" * 70)

print("\nPLAYER FEATURES")
print("-" * 40)

print("Rows:", len(players_fe))
print("Columns:", len(players_fe.columns))

print(
    "New features:",
    [
        "goal_contributions",
        "goal_contributions_per90",
        "shooting_efficiency",
        "shot_accuracy",
        "defensive_actions",
        "defensive_actions_per90",
        "starts_ratio",
        "attacking_output",
        "defensive_output",
        "performance_index"
    ]
)


print("\nTEAM FEATURES")
print("-" * 40)

print("Rows:", len(teams_fe))
print("Columns:", len(teams_fe.columns))

print(
    "New features:",
    [
        "goals_per_shot",
        "shot_accuracy",
        "defensive_actions",
        "attacking_efficiency",
        "defensive_efficiency",
        "team_performance_index"
    ]
)


print("\nMATCH FEATURES")
print("-" * 40)

print("Rows:", len(matches_fe))
print("Columns:", len(matches_fe.columns))

print(
    "New features:",
    [
        "total_goals",
        "goal_difference",
        "result",
        "possession_difference",
        "shot_difference",
        "shots_on_target_difference",
        "total_shots",
        "total_sot",
        "total_fouls",
        "total_yellow_cards",
        "match_intensity"
    ]
)


print("\n" + "=" * 70)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 70)

FEATURE ENGINEERING VALIDATION

PLAYER FEATURES
----------------------------------------
Rows: 1248
Columns: 80
New features: ['goal_contributions', 'goal_contributions_per90', 'shooting_efficiency', 'shot_accuracy', 'defensive_actions', 'defensive_actions_per90', 'starts_ratio', 'attacking_output', 'defensive_output', 'performance_index']

TEAM FEATURES
----------------------------------------
Rows: 48
Columns: 137
New features: ['goals_per_shot', 'shot_accuracy', 'defensive_actions', 'attacking_efficiency', 'defensive_efficiency', 'team_performance_index']

MATCH FEATURES
----------------------------------------
Rows: 104
Columns: 53
New features: ['total_goals', 'goal_difference', 'result', 'possession_difference', 'shot_difference', 'shots_on_target_difference', 'total_shots', 'total_sot', 'total_fouls', 'total_yellow_cards', 'match_intensity']

FEATURE ENGINEERING COMPLETE
